<a href="https://colab.research.google.com/github/CuriousTechNomad/slm-agentic-software-engineering/blob/main/notebooks/00_experiment_protocol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiment Protocol

## Objective

Create the official evaluation subset for all experiments.

This subset will remain fixed throughout the project and will be used for:

- Baseline SLM
- Multi-Agent Workflow
- Fine-Tuned SLM

Using the same tasks ensures reproducibility and fair comparison.

In [1]:
!pip -q install datasets pandas

In [2]:
import json
import random
import pandas as pd
from datasets import load_dataset

In [3]:
dataset = load_dataset("princeton-nlp/SWE-bench_Lite")

df = pd.DataFrame(dataset["test"])

print(df.shape)

README.md:   0%|          | 0.00/3.67k [00:00<?, ?B/s]

data/dev-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  120kB            

data/dev-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.12MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating dev split:   0%|          | 0/23 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/300 [00:00<?, ? examples/s]

(300, 12)


In [4]:
repo_counts = df["repo"].value_counts()

print(repo_counts)

repo
django/django                114
sympy/sympy                   77
scikit-learn/scikit-learn     23
matplotlib/matplotlib         23
pytest-dev/pytest             17
sphinx-doc/sphinx             16
pylint-dev/pylint              6
astropy/astropy                6
psf/requests                   6
pydata/xarray                  5
mwaskom/seaborn                4
pallets/flask                  3
Name: count, dtype: int64


In [5]:
SEED = 42

random.seed(SEED)

In [6]:
selected_rows = []

for repo in df["repo"].unique():

    repo_df = df[df["repo"] == repo]

    if len(repo_df) >= 2:
        selected_rows.append(repo_df.sample(2, random_state=SEED))

    elif len(repo_df) == 1:
        selected_rows.append(repo_df)

evaluation_df = pd.concat(selected_rows)

evaluation_df = evaluation_df.sample(
    n=min(20, len(evaluation_df)),
    random_state=SEED
)

In [7]:
evaluation_df[[
    "instance_id",
    "repo"
]]

,instance_id,repo
147,pallets__flask-4045,pallets/flask
167,pytest-dev__pytest-11143,pytest-dev/pytest
0,astropy__astropy-12907,astropy/astropy
199,scikit-learn__scikit-learn-14894,scikit-learn/scikit-learn
151,psf__requests-2148,psf/requests
148,pallets__flask-4992,pallets/flask
160,pydata__xarray-5131,pydata/xarray
1,astropy__astropy-14182,astropy/astropy
208,sphinx-doc__sphinx-10451,sphinx-doc/sphinx
129,matplotlib__matplotlib-23964,matplotlib/matplotlib


In [10]:
import os

# Create the directory if it doesn't exist
os.makedirs("/content/project_outputs", exist_ok=True)

# Save the evaluation set
evaluation_df.to_json(
    "/content/project_outputs/official_evaluation_set.json",
    orient="records",
    indent=2
)

print("✅ Saved successfully!")

✅ Saved successfully!


In [11]:
ids = evaluation_df["instance_id"].tolist()

with open(
    "/content/project_outputs/evaluation_ids.json",
    "w"
) as f:

    json.dump(ids, f, indent=2)

print(ids)

['pallets__flask-4045', 'pytest-dev__pytest-11143', 'astropy__astropy-12907', 'scikit-learn__scikit-learn-14894', 'psf__requests-2148', 'pallets__flask-4992', 'pydata__xarray-5131', 'astropy__astropy-14182', 'sphinx-doc__sphinx-10451', 'matplotlib__matplotlib-23964', 'django__django-15202', 'pydata__xarray-4094', 'pylint-dev__pylint-6506', 'django__django-11039', 'matplotlib__matplotlib-25079', 'sympy__sympy-12236', 'pytest-dev__pytest-11148', 'sphinx-doc__sphinx-10325', 'sympy__sympy-16792', 'mwaskom__seaborn-3407']


In [12]:
print("Number of Tasks:", len(evaluation_df))

print()

print(evaluation_df["repo"].value_counts())

Number of Tasks: 20

repo
pallets/flask                2
pytest-dev/pytest            2
astropy/astropy              2
pydata/xarray                2
sympy/sympy                  2
sphinx-doc/sphinx            2
matplotlib/matplotlib        2
django/django                2
scikit-learn/scikit-learn    1
psf/requests                 1
pylint-dev/pylint            1
mwaskom/seaborn              1
Name: count, dtype: int64


## Official Evaluation Protocol

Random Seed

42

Evaluation Tasks

20

Selection Strategy

Stratified Random Sampling

Evaluation Dataset

SWE-Bench Lite

These task IDs will be used for all future experiments.